# Phase 2: OpenRouter Multi-Turn Prompting Pipeline

This notebook manages the stateful, multi-turn conversational data collection loop using the OpenRouter API. It ingests the structured problem bank and orchestrates sequential queries to obtain model programs, text explanations, hint hierarchies, and learning summaries.

## Dependencies & Setup
* Initialize OpenAI client pointing to `https://openrouter.ai/api/v1`.
* Configure budget locks, strict warning fallbacks, and rate-limiting (`429`) retry backoffs.

## Dialogue Generation Loop
* **Step 2:** Solution generation constraints (C11 syntax checking, memory safety requirements).
* **Step 3:** Step-by-step instructional explanations.
* **Step 4:** Code-free Socratic hint lattices.
* **Step 5:** Compact learning objective recaps.
* **Step 6:** Automated structural test case definitions.

In [ ]:
import pandas as pd
import json
import os
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Directories and File Paths
BASE_DIR = '/content/drive/MyDrive/LLM_Domain_Results/Supplementary Datasets/'
#FINAL_DATASET_PATH = os.path.join(BASE_DIR, 'Phase2_Sampled_Audit_Dataset.csv')
FINAL_DATASET_PATH = os.path.join(BASE_DIR, 'Phase2_Full_Audit_Dataset.csv')

print("\nPreparing the Master Audit Dataset...")

# 2. Check if we already created the dataset
if os.path.exists(FINAL_DATASET_PATH):
    print(f"Found existing sampled dataset at: {FINAL_DATASET_PATH}")
    master_audit_df = pd.read_csv(FINAL_DATASET_PATH)

    # Ensure no NaN values crash the string formatter
    master_audit_df['problem_text'] = master_audit_df['problem_text'].fillna('').astype(str)

    print("\n==================================================")
    print("MASTER AUDIT QUEUE READY")
    print("==================================================")
    print(f"Loaded {len(master_audit_df)} total problems.")
    print(master_audit_df['source'].value_counts())
else:
    print("Sampled dataset not found. Generating it now from raw files...")

    # Load Mendeley
    mendeley_file = os.path.join(BASE_DIR, 'filtered_mendeley_problems.csv')
    try:
        mendeley_df = pd.read_csv(mendeley_file)
        mendeley_df['source'] = 'Mendeley'
        mendeley_df['global_problem_id'] = 'MD_' + mendeley_df['problem_id'].astype(str)
        mendeley_df['problem_text'] = mendeley_df['problem_description']
        print(f"Loaded {len(mendeley_df)} Mendeley problems. (Keeping all)")
    except FileNotFoundError:
        print(f"Error: Could not find {mendeley_file}")
        mendeley_df = pd.DataFrame()

    # Load LeetCode
    lc_file = os.path.join(BASE_DIR, 'filtered_leetcode_benchmark.jsonl')
    lc_records = []
    try:
        with open(lc_file, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                q_id = data.get('question_id', data.get('id', 'unknown'))
                prob_text = data.get('problem_description', data.get('description', ''))

                lc_records.append({
                    'global_problem_id': f'LC_{q_id}',
                    'source': 'LeetCode',
                    'problem_text': prob_text
                })

        lc_df = pd.DataFrame(lc_records)
        print(f"Loaded {len(lc_df)} raw LeetCode problems.")

    except FileNotFoundError:
        print(f"Error: Could not find {lc_file}")
        lc_df = pd.DataFrame()

    # Merge and Save
    columns_to_keep = ['global_problem_id', 'source', 'problem_text']
    dfs_to_concat = []
    if not mendeley_df.empty: dfs_to_concat.append(mendeley_df[columns_to_keep])
    if not lc_df.empty: dfs_to_concat.append(lc_df[columns_to_keep])

    if dfs_to_concat:
        master_audit_df = pd.concat(dfs_to_concat, ignore_index=True)

        # Shuffle the dataset so Mendeley/LeetCode are mixed evenly
        master_audit_df = master_audit_df.sample(frac=1, random_state=42).reset_index(drop=True)
        master_audit_df['problem_text'] = master_audit_df['problem_text'].fillna('').astype(str)

        # Save to Drive
        master_audit_df.to_csv(FINAL_DATASET_PATH, index=False)
        print("\n==================================================")
        print(f"NEW DATASET SAVED TO: Phase2_Sampled_Audit_Dataset.csv")
        print("==================================================")
        print(f"Total external problems queued for Phase 2: {len(master_audit_df)}")
        print(master_audit_df['source'].value_counts())
    else:
        print("Failed to load datasets. Please check file paths.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Preparing the Master Audit Dataset...
Found existing sampled dataset at: /content/drive/MyDrive/LLM_Domain_Results/Supplementary Datasets/Phase2_Full_Audit_Dataset.csv

MASTER AUDIT QUEUE READY
Loaded 857 total problems.
source
LeetCode    833
Mendeley     24
Name: count, dtype: int64


In [ ]:
!pip install -q openai

import os
from google.colab import userdata
from openai import OpenAI

# Securely retrieve the API key from Colab Secrets
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
# Initialize the client pointing to OpenRouter's base URL
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def call_openrouter(model_name, messages, max_tokens=1500, temperature=0.2):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            extra_body={
                "provider": {
                    "allow_fallbacks": False,
                    "require_parameters": True
                }
            }
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"API_ERROR: {str(e)}"

# --- QUICK CONNECTION TEST ---
print("Testing OpenRouter API connection...")

test_model = "openai/gpt-oss-120b:free" # Using a free model for the test
test_messages = [{"role": "user", "content": "Reply with a single word: Success"}]

test_response = call_openrouter(test_model, test_messages, max_tokens=10)

print("==================================================")
print(f"API Test Output: {test_response}")
print("==================================================")
if "API_ERROR" not in test_response:
    print("API connection is successful and budget locks are active!")
else:
    print("Connection failed. Please check your API key.")

Testing OpenRouter API connection...
API Test Output: Success
API connection is successful and budget locks are active!


In [ ]:
def call_openrouter_with_retry(model_name, messages, max_tokens=3500, temperature=0.2, retries=3):
    """Wraps the API call with a retry loop to handle 429 Rate Limits from free endpoints."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                extra_body={
                    "provider": {
                        "order": ["Groq"],
                        "allow_fallbacks": False,
                        "require_parameters": True
                    }
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            error_str = str(e)
            # If it's a temporary rate limit, wait and retry
            if "429" in error_str and attempt < retries - 1:
                print(f"        [Rate Limited] Waiting 30s before retry {attempt + 1}/{retries}...")
                time.sleep(30)
            # If it's a hard error (like 402 out of budget), fail immediately
            else:
                return f"API_ERROR: {error_str}"

In [ ]:
import os
import time
import pandas as pd

# Define where the final CSV will be saved
OUTPUT_CSV = os.path.join(BASE_DIR, 'GPTOSS_Phase2_External_Audit_Results.csv')
# OUTPUT_CSV = os.path.join(BASE_DIR, 'LLAMA_Phase2_External_Audit_Results.csv')
# OUTPUT_CSV = os.path.join(BASE_DIR, 'KIMIK2_Phase2_External_Audit_Results.csv')

# How often to save to Google Drive
SAVE_EVERY_N = 4

AUDIT_MODELS = [
    "openai/gpt-oss-120b"
    # "meta-llama/llama-3.3-70b-instruct"
    # "moonshotai/kimi-k2-0905"
]

# --- 1. LOAD CHECKPOINT TO RESUME PROGRESS ---
completed_tasks = set()
if os.path.exists(OUTPUT_CSV):
    try:
        # Try standard UTF-8 first
        existing_df = pd.read_csv(OUTPUT_CSV, encoding='utf-8')
    except UnicodeDecodeError:
        print("Detected non-UTF-8 encoding (Excel edit). Adjusting.")
        existing_df = pd.read_csv(OUTPUT_CSV, encoding='cp1252')
    except pd.errors.EmptyDataError:
        existing_df = pd.DataFrame()

    if not existing_df.empty:
        # Create a set of (problem_id, model) tuples that are already finished
        for _, row in existing_df.iterrows():
            completed_tasks.add((row['global_problem_id'], row['model_name']))
        print(f"Resuming from checkpoint: Found {len(completed_tasks)} completed runs.")
    else:
        print("Existing CSV is empty. Starting fresh.")
else:
    print("No existing checkpoint found. Starting fresh.")

# --- 2. PIPELINE DEFINITIONS ---

def get_prompts():
    """Returns the exact prompts used in Phase 1/Phase 2."""
    return [
        """# STEP 2: SOLUTION
        Based on the problem provided in the previous message, provide a complete and correct C solution. Your response must begin with the header # STEP 2: SOLUTION.
        The code must be well-commented to explain the logic of key sections. It must follow modern C standards (e.g., C11), include all necessary headers, and be formatted for readability.
        CRITICAL:
        - The code MUST check the return value of all malloc/realloc calls.
        - All allocated memory MUST be freed before exit.
        - Follow the constraints outlined in the problem.""",

        f"""# STEP 3: EXPLANATION
        Regarding the solution code you just provided, write a clear, step-by-step explanation of how it works. Your response must begin with the header # STEP 3: EXPLANATION.
        Your explanation should be aimed at a student who understands the basic syntax of C but is struggling with the core data structures and memory management concepts required.
        Do not just describe what the code does line-by-line; explain the underlying concepts and the 'why' behind the implementation decisions.""",

        """# STEP 4: HINTS
        Now, imagine a student is stuck on the original problem you created and has not yet seen the solution.
        Generate a series of three progressively more helpful hints to guide them. The hints must not give away the code.
        Your response must begin with the header # STEP 4: HINTS.
        CRITICAL GUARDRAIL: The hints must not give away any actual C code syntax.
        Hint 1: Should be a high-level conceptual nudge about the overall approach.
        Hint 2: Should point them toward a specific part of the problem or a key C feature to use.
        Hint 3: Should be more direct, suggesting a specific logic structure or the first step to take.""",

        """# STEP 5: SUMMARY
        Step 5: Finally, provide a concise summary of the key learning objectives that this problem-solution pair covers.
        The summary should be in bullet-point format and highlight the main C programming concepts a student would master by completing this exercise.
        Your response must begin with the header # STEP 5: SUMMARY.""",

        """# STEP 6: TEST CASES
        For the problem you generated, create a comprehensive suite of 5 test cases. Include at least one common case, one edge case (e.g., empty input, null pointer, zero value), and one invalid input case to test the program's error handling. Your response must begin with the header # STEP 6: TEST CASES.

        CRITICAL FOR AUTOMATION:
        After your descriptions, you MUST provide a machine-readable JSON block.
        containing the raw strings that a user would type to execute these tests. Ensure newlines within the JSON string are represented as literal '\\n' characters and not actual line breaks.

        FOLLOW THIS EXACT STRUCTURE AND FORMAT FOR YOUR JSON BLOCK (i.e., exit_command, test_suite, input, expected_keyword):
        ```json
        {
          "exit_command": "4",
          "test_suite": [
            {"input": "1\\nJohn\\n100", "expected_keyword": "John"},
            {"input": "2\\nJohn", "expected_keyword": "removed"}
          ]
        }
        ```"""
    ]

# --- 3. MAIN EXECUTION LOOP ---
batch_results = []
runs_since_last_save = 0
total_problems = len(master_audit_df)

print("\n==================================================")
print("STARTING CHECKPOINTED AUDIT PIPELINE")
print("==================================================")

for idx, row in master_audit_df.iterrows():
    prob_id = row['global_problem_id']
    source = row['source']
    prob_text = row['problem_text']

    for current_model in AUDIT_MODELS:
        # 1. THE CHECKPOINT SKIP: If already in the memory bank, skip it.
        if (prob_id, current_model) in completed_tasks:
            continue

        print(f"[{idx+1}/{total_problems}] ID: {prob_id} | Src: {source} | Model: {current_model}")

        system_prompt = f"""You are an expert C11 programming tutor.
### PROBLEM DESCRIPTION
{prob_text}
""".strip()

        messages = [{"role": "system", "content": system_prompt}]
        prompts = get_prompts()
        model_failed = False

        # Dictionary to hold the row data for this specific run
        row_data = {
            "global_problem_id": prob_id,
            "source": source,
            "model_name": current_model,
            "problem_text": prob_text
        }

        # Run Steps 2 through 6
        for step_num in range(2, 7):
            prompt_index = step_num - 2
            messages.append({"role": "user", "content": prompts[prompt_index]})

            max_tokens = 2500 if step_num in [2, 6] else 1500

            # Use the API caller
            response_text = call_openrouter_with_retry(current_model, messages, max_tokens=max_tokens)
            if response_text is None:
                response_text = "API_ERROR: Model returned NoneType (Empty Response)"
            messages.append({"role": "assistant", "content": response_text})

            if "API_ERROR" in response_text:
                print(f"    [!] Failed at Step {step_num}. Error: {response_text[:100]}...")
                model_failed = True
                break

            row_data[f"step{step_num}_response"] = response_text

        # If the model completed all steps successfully, stage it for saving
        if not model_failed:
            batch_results.append(row_data)
            completed_tasks.add((prob_id, current_model))
            runs_since_last_save += 1
            print("    [✓] Success")

        # --- THE CHECKPOINT SAVE ---
        if runs_since_last_save >= SAVE_EVERY_N:
            batch_df = pd.DataFrame(batch_results)
            write_header = not os.path.exists(OUTPUT_CSV)
            batch_df.to_csv(OUTPUT_CSV, mode='a', header=write_header, index=False)

            batch_results = []
            runs_since_last_save = 0
            print(f"    ---> Checkpoint saved to Drive!")

if batch_results:
    batch_df = pd.DataFrame(batch_results)
    write_header = not os.path.exists(OUTPUT_CSV)
    batch_df.to_csv(OUTPUT_CSV, mode='a', header=write_header, index=False)
    print(f"    ---> Final Checkpoint saved to Drive!")

print("\n==================================================")
print("EXTERNAL AUDIT PIPELINE FULLY COMPLETE!")
print("==================================================")

Resuming from checkpoint: Found 851 completed runs.

STARTING CHECKPOINTED AUDIT PIPELINE
[250/857] ID: LC_3486 | Src: LeetCode | Model: openai/gpt-oss-120b
        [Rate Limited] Waiting 30s before retry 1/3...
    [!] Failed at Step 2. Error: API_ERROR: Model returned NoneType (Empty Response)...
[373/857] ID: LC_2532 | Src: LeetCode | Model: openai/gpt-oss-120b
    [!] Failed at Step 2. Error: API_ERROR: Model returned NoneType (Empty Response)...
[460/857] ID: LC_3420 | Src: LeetCode | Model: openai/gpt-oss-120b
    [!] Failed at Step 2. Error: API_ERROR: Model returned NoneType (Empty Response)...
[561/857] ID: LC_3017 | Src: LeetCode | Model: openai/gpt-oss-120b
    [!] Failed at Step 2. Error: API_ERROR: Model returned NoneType (Empty Response)...
[646/857] ID: LC_2972 | Src: LeetCode | Model: openai/gpt-oss-120b
        [Rate Limited] Waiting 30s before retry 1/3...
    [!] Failed at Step 2. Error: API_ERROR: Model returned NoneType (Empty Response)...
[788/857] ID: LC_3302 | S